# Pandas Revision — The `mpg` Dataset

In [2]:
import pandas as pd
import numpy as np

cars = pd.read_csv("mpg.csv")
cars.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


## Q1. Inspect the data

Before touching anything, find out what you are holding.

1. How many rows and columns?
2. What is the dtype of each column?
3. Which columns have missing values, and how many?

In [3]:
# Q1 — Solution

# 1) Shape gives (number_of_rows, number_of_columns).
print("Rows and columns:", cars.shape)

# 2) dtypes tells us the data type of every column.
print("\nData types:")
print(cars.dtypes)

# 3) isna().sum() counts missing values in each column.
print("\nMissing values:")
print(cars.isna().sum())

Rows and columns: (398, 9)

Data types:
mpg             float64
cylinders         int64
displacement    float64
horsepower      float64
weight            int64
acceleration    float64
model_year        int64
origin              str
name                str
dtype: object

Missing values:
mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64


## Q2. Select columns

Show only `name`, `mpg` and `origin`, for the first 5 rows.

In [4]:
# Q2 — Solution
# Use a list of column names to select several columns.
# .head(5) keeps only the first five rows.
cars[["name", "mpg", "origin"]].head(5)

,name,mpg,origin
0,chevrolet chevelle malibu,18.0,usa
1,buick skylark 320,15.0,usa
2,plymouth satellite,18.0,usa
3,amc rebel sst,16.0,usa
4,ford torino,17.0,usa


## Q3. Filter rows

How many cars manage more than 30 mpg? Of those, how many are Japanese?

In [ ]:
# Q3 — Solution

# First create a Boolean filter for mpg > 30.
over_30 = cars[cars["mpg"] > 30]

# len(...) counts the rows that passed the filter.
total_over_30 = len(over_30)

# Then filter those rows to Japanese cars.
japanese_over_30 = over_30[over_30["origin"].astype(str).str.lower().eq("japan")]

print("Cars with mpg > 30:", total_over_30)
print("Japanese cars among them:", len(japanese_over_30))

# Display the matching Japanese cars as a useful check.
japanese_over_30[["name", "mpg", "origin"]]

Cars with mpg > 30: 85
Japanese cars among them: 46


,name,mpg,origin
53,toyota corolla 1200,31.0,japan
54,datsun 1200,35.0,japan
129,datsun b210,31.0,japan
131,toyota corolla 1200,32.0,japan
144,toyota corona,31.0,japan
145,datsun 710,32.0,japan
181,honda civic cvcc,33.0,japan
198,honda civic,33.0,japan
204,datsun b-210,32.0,japan
216,honda accord cvcc,31.5,japan


## Q4. Handle missing values

`horsepower` has blanks.

1. Which origins do the missing rows belong to?
2. Fill the blanks with the median horsepower.
3. Confirm there are no blanks left.

### Why the median?
The median is less affected by unusually high or low horsepower values than the mean.

In [6]:
# Q4 — Solution

# Find rows where horsepower is missing.
missing_hp = cars[cars["horsepower"].isna()]

# 1) See which origins those missing rows belong to.
print("Origins of rows with missing horsepower:")
print(missing_hp["origin"].value_counts(dropna=False))

# 2) Convert horsepower to numeric in case the CSV stored it as text.
# Errors become NaN, which is appropriate for missing/non-numeric entries.
cars["horsepower"] = pd.to_numeric(cars["horsepower"], errors="coerce")

# Calculate the median using only the non-missing horsepower values.
hp_median = cars["horsepower"].median()
print("\nMedian horsepower:", hp_median)

# Fill missing horsepower values with the median.
cars["horsepower"] = cars["horsepower"].fillna(hp_median)

# 3) Confirm there are no missing horsepower values.
print("\nMissing horsepower after filling:", cars["horsepower"].isna().sum())

Origins of rows with missing horsepower:
origin
usa       4
europe    2
Name: count, dtype: int64

Median horsepower: 93.5

Missing horsepower after filling: 0


## Q5. Create a new column

Power-to-weight ratio is a better predictor of how a car feels than horsepower alone.

Add `power_to_weight = horsepower ÷ weight`. Then show the 5 cars with the highest ratio.

In [7]:
# Q5 — Solution

# Divide horsepower by weight row-by-row.
cars["power_to_weight"] = cars["horsepower"] / cars["weight"]

# Sort from largest ratio to smallest and take the first five.
top_5_power_to_weight = (
    cars.sort_values("power_to_weight", ascending=False)
        [["name", "horsepower", "weight", "power_to_weight"]]
        .head(5)
)

top_5_power_to_weight

,name,horsepower,weight,power_to_weight
13,buick estate wagon (sw),225.0,3086,0.072910
116,pontiac grand prix,230.0,4278,0.053763
330,renault lecar deluxe,93.5,1835,0.050954
8,pontiac catalina,225.0,4425,0.050847
23,bmw 2002,113.0,2234,0.050582


## Q6. Count with `value_counts`

1. How many cars come from each origin?
2. How many cars have each cylinder count?
3. Show the origin breakdown as percentages instead of counts.

In [8]:
# Q6 — Solution

# 1) Count cars by origin.
print("Cars by origin:")
print(cars["origin"].value_counts())

# 2) Count cars by number of cylinders.
print("\nCars by cylinder count:")
print(cars["cylinders"].value_counts().sort_index())

# 3) normalize=True returns proportions rather than raw counts.
# Multiplying by 100 converts proportions into percentages.
print("\nOrigin breakdown (%):")
print((cars["origin"].value_counts(normalize=True) * 100).round(1))

Cars by origin:
origin
usa       249
japan      79
europe     70
Name: count, dtype: int64

Cars by cylinder count:
cylinders
3      4
4    204
5      3
6     84
8    103
Name: count, dtype: int64

Origin breakdown (%):
origin
usa       62.6
japan     19.8
europe    17.6
Name: proportion, dtype: float64


## Q7. Named aggregation

Build one table with a row per origin and these columns:

- `cars`
- `avg_mpg`
- `avg_weight`
- `avg_cylinders`

Round to 1 decimal.

In [9]:
# Q7 — Solution

# groupby("origin") creates one group for each origin.
# Named aggregation lets us choose clear output column names.
origin_summary = (
    cars.groupby("origin")
        .agg(
            cars=("name", "size"),
            avg_mpg=("mpg", "mean"),
            avg_weight=("weight", "mean"),
            avg_cylinders=("cylinders", "mean")
        )
        .round(1)
)

origin_summary

,cars,avg_mpg,avg_weight,avg_cylinders
origin,,,,
europe,70,27.9,2423.3,4.2
japan,79,30.5,2221.2,4.1
usa,249,20.1,3361.9,6.2


## Q8. Size vs count

For each origin, show how many cars there are **and** how many have a horsepower recorded.

The first number is the total number of rows. The second counts non-missing horsepower values.

In [10]:
# Q8 — Solution

origin_size_vs_hp = (
    cars.groupby("origin")
        .agg(
            cars=("name", "size"),
            horsepower_recorded=("horsepower", "count")
        )
)

origin_size_vs_hp

,cars,horsepower_recorded
origin,,
europe,70,70
japan,79,79
usa,249,249


## Q9. Group by two columns

Calculate average mpg for every combination of `origin` and `cylinders`.

Then reshape it with `.unstack()` so origins are rows and cylinder counts are columns.

In [11]:
# Q9 — Solution

# Group by two columns and calculate the mean mpg.
avg_mpg_by_origin_cylinders = cars.groupby(
    ["origin", "cylinders"]
)["mpg"].mean()

# unstack() turns the second index level (cylinders)
# into columns, creating a readable cross-tab style table.
avg_mpg_table = avg_mpg_by_origin_cylinders.unstack().round(1)

avg_mpg_table

cylinders,3,4,5,6,8
origin,,,,,
europe,NaN,28.4,27.4,20.1,NaN
japan,20.6,31.6,NaN,23.9,NaN
usa,NaN,27.8,NaN,19.7,15.0


## Q10. The confound

Someone shows you:

> Japanese cars average 30.5 mpg. American cars average 20.1 mpg.  
> Japanese engineering is simply better at fuel efficiency.

Both averages are correct. Is the conclusion correct?

**Investigate. Compare like with like, and state clearly what the data does and does not support.**

### Key idea: confounding
An overall average can hide an important third variable. Here, **cylinder count** is a natural candidate because engines with more cylinders generally differ in size and fuel consumption.

In [12]:
# Q10 — Solution

# First reproduce the overall averages.
overall = cars.groupby("origin")["mpg"].mean().round(1)
print("Overall average mpg:")
print(overall)

# Now compare American and Japanese cars within the SAME cylinder counts.
# This is a fairer "like with like" comparison.
like_with_like = (
    cars[cars["origin"].isin(["usa", "japan"])]
    .groupby(["cylinders", "origin"])["mpg"]
    .mean()
    .unstack()
    .round(1)
)

print("\nAverage mpg by cylinder count and origin:")
print(like_with_like)

# Also compare how the two origins are distributed across cylinder counts.
cylinder_mix = pd.crosstab(cars["cylinders"], cars["origin"], normalize="columns") * 100
print("\nCylinder mix within each origin (%):")
print(cylinder_mix.round(1))

print("\nInterpretation:")
print(
    "The overall averages alone do NOT prove that Japanese engineering is "
    "simply better. If Japan and the USA have different mixes of cylinder "
    "counts, the overall averages combine different kinds of cars. "
    "The within-cylinder comparison is the relevant evidence for a direct "
    "like-for-like comparison."
)

Overall average mpg:
origin
europe    27.9
japan     30.5
usa       20.1
Name: mpg, dtype: float64

Average mpg by cylinder count and origin:
origin     japan   usa
cylinders             
3           20.6   NaN
4           31.6  27.8
6           23.9  19.7
8            NaN  15.0

Cylinder mix within each origin (%):
origin     europe  japan   usa
cylinders                     
3             0.0    5.1   0.0
4            90.0   87.3  28.9
5             4.3    0.0   0.0
6             5.7    7.6  29.7
8             0.0    0.0  41.4

Interpretation:
The overall averages alone do NOT prove that Japanese engineering is simply better. If Japan and the USA have different mixes of cylinder counts, the overall averages combine different kinds of cars. The within-cylinder comparison is the relevant evidence for a direct like-for-like comparison.


## Q11. `idxmax`

Which specific car has the best fuel economy in each origin?

Show the car's **name and mpg**, not just the number.

In [13]:
# Q11 — Solution

# idxmax() returns the index label of the row containing the maximum mpg
# within each origin.
best_idx = cars.groupby("origin")["mpg"].idxmax()

best_cars = (
    cars.loc[best_idx, ["origin", "name", "mpg"]]
        .sort_values("origin")
        .reset_index(drop=True)
)

best_cars

,origin,name,mpg
0,europe,vw rabbit c (diesel),44.3
1,japan,mazda glc,46.6
2,usa,plymouth champ,39.0


## Q12. `astype` and `to_csv`

1. Convert `origin` to the `category` dtype and report the memory saved.
2. Save your Q7 summary table to `origin_summary.csv` — with the origin column intact.
3. Read the file back and confirm nothing was lost.

### Why `category`?
A categorical column stores repeated labels more efficiently when there are only a small number of unique values.

In [14]:
# Q12 — Solution

# Measure memory before conversion.
memory_before = cars["origin"].memory_usage(deep=True)

# Convert origin to categorical data.
cars["origin"] = cars["origin"].astype("category")

# Measure memory after conversion.
memory_after = cars["origin"].memory_usage(deep=True)

print("Memory before:", memory_before, "bytes")
print("Memory after: ", memory_after, "bytes")
print("Memory saved: ", memory_before - memory_after, "bytes")

# Make the origin an ordinary column before writing the summary.
# index=False prevents pandas from creating an extra unnamed index column.
origin_summary.to_csv("origin_summary.csv", index=True)

# Read the file back.
summary_back = pd.read_csv("origin_summary.csv", index_col=0)

print("\nSummary read back from CSV:")
print(summary_back)

# Confirm shape and column names match the original summary.
print("\nShape unchanged:", summary_back.shape == origin_summary.shape)
print("Columns unchanged:", list(summary_back.columns) == list(origin_summary.columns))

Memory before: 21196 bytes
Memory after:  691 bytes
Memory saved:  20505 bytes

Summary read back from CSV:
        cars  avg_mpg  avg_weight  avg_cylinders
origin                                          
europe    70     27.9      2423.3            4.2
japan     79     30.5      2221.2            4.1
usa      249     20.1      3361.9            6.2

Shape unchanged: True
Columns unchanged: True


## Q13. Sorting

Show the 10 heaviest cars, with `name`, `weight` and `mpg`.

What do you notice about their mpg?

In [15]:
# Q13 — Solution

heaviest_10 = (
    cars.sort_values("weight", ascending=False)
        [["name", "weight", "mpg"]]
        .head(10)
)

heaviest_10

,name,weight,mpg
44,pontiac safari (sw),5140,13.0
103,chevrolet impala,4997,11.0
42,dodge monaco (sw),4955,12.0
90,mercury marquis brougham,4952,12.0
95,buick electra 225 custom,4951,12.0
104,ford country,4906,12.0
43,ford country squire (sw),4746,13.0
94,chrysler new yorker brougham,4735,13.0
28,hi 1200d,4732,9.0
137,buick century luxus (sw),4699,13.0


**Observation:** The heaviest cars generally have relatively low mpg. This is a descriptive pattern in this dataset, not by itself proof of causation.

## Q14. Compound filter

Find American cars that:

- are 4-cylinder,
- are from model year 1978 or later,
- manage more than 25 mpg.

How many are there? Show the 5 most efficient.

In [16]:
# Q14 — Solution

# Each condition is wrapped in parentheses because & combines Boolean Series.
american_efficient_4cyl = cars[
    (cars["origin"].astype(str).str.lower() == "usa")
    & (cars["cylinders"] == 4)
    & (cars["model_year"] >= 78)
    & (cars["mpg"] > 25)
]

print("Number of matching cars:", len(american_efficient_4cyl))

# Sort by mpg descending to get the five most efficient.
top_5 = (
    american_efficient_4cyl
    .sort_values("mpg", ascending=False)
    [["name", "model_year", "cylinders", "mpg", "origin"]]
    .head(5)
)

top_5

Number of matching cars: 37


,name,model_year,cylinders,mpg,origin
344,plymouth champ,81,4,39.0,usa
378,plymouth horizon miser,82,4,38.0,usa
245,ford fiesta,78,4,36.1,usa
391,dodge charger 2.2,82,4,36.0,usa
379,mercury lynx l,82,4,36.0,usa


## Q15. String methods

The `name` column holds make and model together, such as `chevrolet chevelle malibu`.

1. Extract the brand — the first word — into a new `brand` column.
2. List every distinct brand.
3. Something is wrong with that list. Find the problem and fix it.

### Important
The classic Auto MPG data contains a few spelling variants/typos in manufacturer names. We standardize those before using the brand column for analysis.

In [17]:
# Q15 — Solution

# 1) The first word is the brand.
# split() breaks each name into words; str[0] takes the first word.
cars["brand"] = cars["name"].str.split().str[0]

print("Raw distinct brands:")
print(sorted(cars["brand"].dropna().unique()))

# 2) Inspect suspicious/rare spellings. The exact output can vary slightly
# depending on the version of mpg.csv being used.
print("\nBrand frequencies:")
print(cars["brand"].value_counts().sort_index())

# 3) Standardize known spelling variants found in the Auto MPG dataset.
# This dictionary is deliberately easy to extend if your local CSV contains
# another obvious typo.
brand_fixes = {
    "maxda": "mazda",
    "mercedes-benz": "mercedes-benz",
    "toyouta": "toyota",
    "vokswagen": "volkswagen",
    "vw": "volkswagen",
}

cars["brand"] = cars["brand"].replace(brand_fixes)

print("\nCleaned distinct brands:")
print(sorted(cars["brand"].dropna().unique()))

Raw distinct brands:
['amc', 'audi', 'bmw', 'buick', 'cadillac', 'capri', 'chevroelt', 'chevrolet', 'chevy', 'chrysler', 'datsun', 'dodge', 'fiat', 'ford', 'hi', 'honda', 'maxda', 'mazda', 'mercedes', 'mercedes-benz', 'mercury', 'nissan', 'oldsmobile', 'opel', 'peugeot', 'plymouth', 'pontiac', 'renault', 'saab', 'subaru', 'toyota', 'toyouta', 'triumph', 'vokswagen', 'volkswagen', 'volvo', 'vw']

Brand frequencies:
brand
amc              28
audi              7
bmw               2
buick            17
cadillac          2
capri             1
chevroelt         1
chevrolet        43
chevy             3
chrysler          6
datsun           23
dodge            28
fiat              8
ford             51
hi                1
honda            13
maxda             2
mazda            10
mercedes          1
mercedes-benz     2
mercury          11
nissan            1
oldsmobile       10
opel              4
peugeot           8
plymouth         31
pontiac          16
renault           5
saab            

## Q16. Brand summary

Using your cleaned `brand` column:

- Keep every brand with at least 15 cars.
- Show the number of cars.
- Show the average mpg.
- Sort from most efficient to least efficient.

In [18]:
# Q16 — Solution

brand_summary = (
    cars.groupby("brand")
        .agg(
            cars=("name", "size"),
            avg_mpg=("mpg", "mean")
        )
        # Keep only brands represented by at least 15 cars.
        .query("cars >= 15")
        # Highest average mpg first = most efficient first.
        .sort_values("avg_mpg", ascending=False)
        .round({"avg_mpg": 1})
)

brand_summary

,cars,avg_mpg
brand,,
volkswagen,22,31.8
datsun,23,31.1
toyota,26,28.2
dodge,28,22.1
plymouth,31,21.7
chevrolet,43,20.5
pontiac,16,20.0
ford,51,19.7
buick,17,19.2


## Q17. Change over time

1. What was the average mpg of the 1970 fleet? The 1982 fleet?
2. Someone concludes: **"Japanese cars got much more efficient over the decade — that is why they took the market."**
   Check it. Did Japanese cars improve more than everyone else's?

### Important
`model_year` uses two-digit years:
- `70` = 1970
- `82` = 1982

In [19]:
# Q17 — Solution

# 1) Overall fleet averages for 1970 and 1982.
fleet_1970 = cars.loc[cars["model_year"] == 70, "mpg"].mean()
fleet_1982 = cars.loc[cars["model_year"] == 82, "mpg"].mean()

print("1970 fleet average mpg:", round(fleet_1970, 1))
print("1982 fleet average mpg:", round(fleet_1982, 1))

# 2) Compare improvement by origin.
# We calculate the mean mpg for each origin at the two endpoints.
change = (
    cars[cars["model_year"].isin([70, 82])]
    .groupby(["origin", "model_year"])["mpg"]
    .mean()
    .unstack()
)

change["improvement"] = change[82] - change[70]

print("\nAverage mpg in 1970 and 1982 by origin:")
print(change.round(1))

print("\nInterpretation:")
print(
    "To support the claim that Japanese cars improved MORE than everyone else, "
    "their increase from 1970 to 1982 would need to be larger than the "
    "corresponding increases for the other origins. The table above lets us "
    "compare those changes directly."
)

1970 fleet average mpg: 17.7
1982 fleet average mpg: 31.7

Average mpg in 1970 and 1982 by origin:
model_year    70    82  improvement
origin                             
europe      25.2  40.0         14.8
japan       25.5  34.9          9.4
usa         15.3  29.4         14.2

Interpretation:
To support the claim that Japanese cars improved MORE than everyone else, their increase from 1970 to 1982 would need to be larger than the corresponding increases for the other origins. The table above lets us compare those changes directly.


## Q18. `idxmin`

For each cylinder count, which car has the **worst fuel economy**?

Because higher mpg is better, "worst fuel economy" means the **minimum mpg** within each cylinder group.

In [ ]:
# Q18 — Solution

# idxmin() returns the row index containing the smallest mpg
# within each cylinder-count group.
worst_idx = cars.groupby("cylinders")["mpg"].idxmin()

worst_cars = (
    cars.loc[worst_idx, ["cylinders", "name", "mpg"]].sort_values("cylinders").reset_index(drop=True)
)

worst_cars

,cylinders,name,mpg
0,3,maxda rx3,18.0
1,4,volvo 145e (sw),18.0
2,5,audi 5000,20.3
3,6,chevrolet nova,15.0
4,8,hi 1200d,9.0


# Bonus: Is acceleration correlated with mpg?

The assignment notes an important detail:

> Lower acceleration numbers mean faster cars.

We can calculate the correlation, but remember that correlation describes association, not causation.

In [21]:
# Bonus — Solution

correlation = cars[["acceleration", "mpg"]].corr().loc["acceleration", "mpg"]

print("Correlation between acceleration and mpg:", round(correlation, 3))

print(
    "\nInterpretation: Because acceleration is measured in seconds, a lower "
    "acceleration value means a faster 0–60 time. Therefore, a negative "
    "correlation would mean faster cars tend to have higher mpg in this dataset, "
    "while a positive correlation would mean faster cars tend to have lower mpg."
)

Correlation between acceleration and mpg: 0.42

Interpretation: Because acceleration is measured in seconds, a lower acceleration value means a faster 0–60 time. Therefore, a negative correlation would mean faster cars tend to have higher mpg in this dataset, while a positive correlation would mean faster cars tend to have lower mpg.


## The two most important reasoning questions

**Q10:** Do not trust an overall average until you ask whether the groups are comparable. Check potential confounders such as cylinder count.

**Q17:** Compare the *change* over time for each origin, not just the starting or ending averages. A higher final average does not automatically mean a larger improvement.